# Phase 2: The Math of Prediction
## From the Chain Rule to N-Grams

In Phase 1, we saw *that* words have tags. Now, we ask: **How does a model predict the next word or tag?**

### 1. The Chain Rule of Probability
To compute the probability of a full sentence $P(W)$, we use the **Chain Rule**. This calculates the likelihood of each word given *all* previous words.

**Example:** 
$P(\text{"I want to eat"}) = P(\text{"I"}) \times P(\text{"want"}|\text{"I"}) \times P(\text{"to"}|\text{"I want"}) \times P(\text{"eat"}|\text{"I want to"})$.

### 2. The Markov Assumption: Simplifying the History
Calculating the probability based on the *entire* history is computationally impossible—we'll never see enough data. 

**The Solution:** We approximate by looking back only $k$ steps. 
* **Unigram ($k=0$):** Probability depends only on the word itself.
* **Bigram ($k=1$):** Probability depends on the current word and the *one* previous word.

#### **Visualizing Contextual Look-back**
Watch the video below. Notice how the 'glow' highlights only the immediate previous words to predict the next one. This is the Markov Assumption in action.

In [1]:
from IPython.display import Video
# Ensure your Veo video is named exactly 'PredictiveText.mp4'
Video("./video/Digital_Cursor_Typing_Chinese_Food.mp4", width=600)

**Note** - This video was generated using this prompt:

A cinematic, close-up video of a digital cursor typing the sentence 'I want to eat...'. As it types, translucent layers of previous words highlight and fade, showing the model 'looking back' at 'want' and 'to' before the word 'Chinese' appears. High-tech HUD interface style.

### 3. Activity: The Bigram Prediction Engine
Let's build a simple Bigram model using the **Maximum Likelihood Estimate (MLE)** formula[cite: 70, 71]:

$$P(w_i | w_{i-1}) = \frac{count(w_{i-1}, w_i)}{count(w_{i-1})}$$

**Task:** Run the cell below to train a model on a tiny corpus and predict the next word.

In [2]:
from collections import Counter, defaultdict

# Our tiny training corpus [cite: 73, 74, 75]
corpus = "I am Sam . Sam I am . I do not like red eggs and ham ."
tokens = corpus.split()

# Count Unigrams and Bigrams
unigram_counts = Counter(tokens)
bigrams = list(zip(tokens, tokens[1:]))
bigram_counts = Counter(bigrams)

def get_bigram_probability(prev_word, current_word):
    count_prev = unigram_counts[prev_word]
    count_bigram = bigram_counts[(prev_word, current_word)]
    
    if count_prev == 0: return 0
    return count_bigram / count_prev

# Predict: What follows 'I'?
options = ["am", "do", "Sam"]
for opt in options:
    prob = get_bigram_probability("I", opt)
    print(f"P({opt} | I) = {prob:.2f}")

P(am | I) = 0.67
P(do | I) = 0.33
P(Sam | I) = 0.00


The formula being executed in your Python function is:

$$P(w_i \mid w_{i-1}) = \frac{\text{count}(w_{i-1}, w_i)}{\text{count}(w_{i-1})}$$

Based on the corpus provided, here is the converted Markdown table:

### Bigram Probability Calculations
**Corpus:** "I am Sam . Sam I am . I do not like red eggs and ham ."

| Bigram | Reasoning | Result |
| :--- | :--- | :--- |
| $P(am \mid I)$ | $\frac{\text{count("I am")}}{\text{count("I")}}$ | $2 / 3$ |
| $P(do \mid I)$ | $\frac{\text{count("I do")}}{\text{count("I")}}$ | $1 / 3$ |
| $P(Sam \mid I)$ | $\frac{\text{count("I Sam")}}{\text{count("I")}}$ | $0 / 3$ |

### 4. Critical Discussion: Long-Distance Dependencies
N-Gram models are often 'sufficient,' but they are linguistically 'insufficient'. 

**Question for the class:** In the sentence: *"The computer which I had just put into the machine room on the fifth floor **crashed**."*

Can a Bigram model predict that 'crashed' belongs to 'computer'? Why or why not?

**Not really!**

This demonstrates why we "never see enough data" to estimate all possible word combinations accurately. This is exactly why we need Smoothing (Phase 3) to "steal" a little bit of probability from the common words and give it to the zeros.